# PPE Detection v2 – improvement experiments
**Do not edit any code.** Run the cells in order (Runtime → Run all) after choosing **Runtime → Change runtime type → T4 GPU → Save**.

What this does (the baseline v1 was YOLO11s at 640 px, test mAP@50 = 0.602):
1. Trains two candidates at **960 px** (YOLO11s and YOLO11m). Many objects in your images are small, so higher resolution is the main thing being tested.
2. Picks the best on the **validation** set only. If neither beats v1 on validation, it stops and does **not** touch the test set.
3. Chooses a confidence threshold on validation (recall-weighted).
4. Evaluates the chosen model on the **test** set once, and downloads a ZIP.

Expected time: probably 1.5–3 hours on a T4 (not measured yet). Keep this tab open. The only click needed is **Choose files** in Step 3.

## Step 1 – Check the GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU OK:', torch.cuda.get_device_name(0))
else:
    print('NO GPU FOUND. Runtime > Change runtime type > T4 GPU > Save, then run this cell again.')
    raise SystemExit('Stopped: GPU is required for training.')

## Step 2 – Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

## Step 3 – Choose your dataset ZIP
Click **Choose files** and select `ppe_4classes.zip` (about 80 MB). Wait for 100%.

In [ ]:
import os, glob, zipfile, shutil

ZIP = '/content/ppe_4classes.zip'

# 1) Get the ZIP (skip the upload if it is already in /content)
if not os.path.exists(ZIP):
    from google.colab import files
    print('Click "Choose files" below and select ppe_4classes.zip ...')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file selected. Run this cell again and choose ppe_4classes.zip.')
    name = list(uploaded.keys())[0]
    if os.path.abspath(name) != ZIP:
        shutil.move(name, ZIP)

if not zipfile.is_zipfile(ZIP):
    os.remove(ZIP)
    raise SystemExit('That file is not a valid ZIP. Run this cell again and choose ppe_4classes.zip.')

# 2) Extract fresh
shutil.rmtree('/content/ds', ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/ds')

# 3) Find the dataset folder automatically (the one containing train/valid/test)
DS = None
for y in glob.glob('/content/ds/**/data.yaml', recursive=True):
    d = os.path.dirname(y)
    if all(os.path.isdir(f'{d}/{s}/images') for s in ('train', 'valid', 'test')):
        DS = d
        break
if DS is None:
    raise SystemExit('Could not find train/valid/test folders inside the ZIP. Make sure you chose ppe_4classes.zip.')

# 4) Verify image/label pairing and count annotations
expected = {'train': (521, 1385), 'valid': (114, 295), 'test': (82, 302)}
names = ['Hardhat_Worn', 'Hardhat_Not_Worn', 'Safety_Vest_Worn', 'Safety_Vest_Not_Worn']
all_ok = True
for s, (ei, ea) in expected.items():
    imgs = sorted(os.listdir(f'{DS}/{s}/images'))
    lbls = sorted(os.listdir(f'{DS}/{s}/labels'))
    stems_i = {os.path.splitext(f)[0] for f in imgs}
    stems_l = {os.path.splitext(f)[0] for f in lbls}
    n_ann, bad = 0, 0
    for f in lbls:
        for line in open(f'{DS}/{s}/labels/{f}').read().splitlines():
            if not line.strip():
                continue
            p = line.split()
            if len(p) != 5 or int(p[0]) not in range(4):
                bad += 1
            n_ann += 1
    ok = (len(imgs) == ei and n_ann == ea and stems_i == stems_l and bad == 0)
    all_ok &= ok
    print(f'{s:6s} images={len(imgs)} (expected {ei})  annotations={n_ann} (expected {ea})  '
          f'paired={stems_i == stems_l}  bad_lines={bad}  ->', 'OK' if ok else 'PROBLEM')
if not all_ok:
    raise SystemExit('Dataset check failed - see the PROBLEM line above and tell Claude.')

# 5) Write the YOLO data file with the correct absolute path (no manual editing needed)
with open('/content/data.yaml', 'w') as f:
    f.write(f'path: {DS}\ntrain: train/images\nval: valid/images\ntest: test/images\n\nnc: 4\nnames:\n')
    for i, n in enumerate(names):
        f.write(f'  {i}: {n}\n')
print('\nDataset verified and ready. data.yaml written.')

## Step 4 – Train the two candidates (longest step)

In [ ]:
import time, json, os, gc, torch
from ultralytics import YOLO

# Variables tested: model size and input resolution (baseline v1 = yolo11s @ 640).
# Everything else is fixed. Selection uses the VALIDATION set only.
EXPERIMENTS = [
    dict(name='v2a_yolo11s_960', model='yolo11s.pt', imgsz=960, batches=[16, 8]),
    dict(name='v2b_yolo11m_960', model='yolo11m.pt', imgsz=960, batches=[8, 4]),
]
COMMON = dict(epochs=80, patience=25, seed=0, deterministic=True, workers=2)
DATA = '/content/data.yaml'
RUNS = '/content/runs_v2'
summary = []

for ex in EXPERIMENTS:
    t0 = time.time(); trained = False
    for b in ex['batches']:
        try:
            print(f"\n=== {ex['name']}: {ex['model']} imgsz={ex['imgsz']} batch={b} ===")
            YOLO(ex['model']).train(data=DATA, imgsz=ex['imgsz'], batch=b, project=RUNS,
                                    name=ex['name'], exist_ok=True, **COMMON)
            trained = True; break
        except torch.cuda.OutOfMemoryError:
            print('Out of GPU memory, retrying with a smaller batch...')
            gc.collect(); torch.cuda.empty_cache()
    if not trained:
        print('Run failed:', ex['name']); continue
    best_path = f"{RUNS}/{ex['name']}/weights/best.pt"
    v = YOLO(best_path).val(data=DATA, split='val', imgsz=ex['imgsz'], plots=False, verbose=False)
    row = dict(name=ex['name'], model=ex['model'], imgsz=ex['imgsz'], batch=b, minutes=round((time.time()-t0)/60, 1),
               val_precision=float(v.box.mp), val_recall=float(v.box.mr), val_map50=float(v.box.map50), val_map50_95=float(v.box.map),
               best_path=best_path)
    summary.append(row)
    print({k: (round(x, 3) if isinstance(x, float) else x) for k, x in row.items()})
    os.makedirs('/content/results_v2', exist_ok=True)
    json.dump(summary, open('/content/results_v2/experiments_summary.json', 'w'), indent=1)

print('\nBaseline v1 (yolo11s @640) validation: P=0.879 R=0.567 mAP50=0.663 mAP50-95=0.410')
for r in summary:
    print(f"{r['name']:18s} val mAP50={r['val_map50']:.3f} mAP50-95={r['val_map50_95']:.3f} P={r['val_precision']:.3f} R={r['val_recall']:.3f}  ({r['minutes']} min)")

## Step 5 – Select the winner on validation only

In [ ]:
import json
BASE_VAL = dict(map50=0.663, map50_95=0.410)   # real v1 validation values from the first run
assert summary, 'No experiment finished. Tell Claude what the error above says.'
winner = max(summary, key=lambda r: (r['val_map50_95'], r['val_map50']))
print('Best new model on VALIDATION:', winner['name'], f"mAP50-95={winner['val_map50_95']:.3f} mAP50={winner['val_map50']:.3f}")
IMPROVED = winner['val_map50_95'] > BASE_VAL['map50_95']
if IMPROVED:
    print('This beats the v1 baseline on validation (0.410 mAP50-95). Continuing to threshold selection and the single test evaluation.')
else:
    print('No new model beat the v1 baseline on validation. The test set will NOT be touched. Keep v1 and tell Claude.')

## Step 6 – Choose the confidence threshold on validation

In [ ]:
import numpy as np
from ultralytics import YOLO
assert IMPROVED, 'Stopped: no improvement on validation (see the previous step).'
WBEST, IMGSZ = winner['best_path'], winner['imgsz']
bw = YOLO(WBEST)
vv = bw.val(data=DATA, split='val', imgsz=IMGSZ, conf=0.001, plots=False, verbose=False)
px, P, R = vv.box.px, vv.box.p_curve, vv.box.r_curve          # per-class curves over confidence
beta = 2.0                                                     # F2: recall counts more than precision (missed violations are costly)
F2 = ((1 + beta**2) * P * R / (beta**2 * P + R + 1e-9)).mean(0)
F1 = (2 * P * R / (P + R + 1e-9)).mean(0)
ok = (px >= 0.05) & (px <= 0.5)
CONF = float(px[ok][int(np.argmax(F2[ok]))])
CONF_F1 = float(px[ok][int(np.argmax(F1[ok]))])
print(f'Chosen confidence threshold (max macro-F2 on VALIDATION): {CONF:.3f}   (max macro-F1 would be {CONF_F1:.3f})')
print('conf   val macro-P  val macro-R')
for c in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, CONF]:
    i = int(np.argmin(np.abs(px - c))); print(f'{c:.3f}  {P[:, i].mean():.3f}        {R[:, i].mean():.3f}')

## Step 7 – Evaluate the winner on the test set (once)

In [ ]:
import os, csv, json, numpy as np
from ultralytics import YOLO
RES = '/content/results_v2'; os.makedirs(RES, exist_ok=True)
best = YOLO(WBEST)

# Single test evaluation of the chosen model (low conf so full PR curves exist)
t = best.val(data=DATA, split='test', imgsz=IMGSZ, conf=0.001, plots=True, project=RES, name='test_eval', exist_ok=True)
print('TEST (v2) mAP50=%.3f mAP50-95=%.3f   P=%.3f R=%.3f (at the max-F1 point)' % (t.box.map50, t.box.map, t.box.mp, t.box.mr))

px, P, R = t.box.px, t.box.p_curve, t.box.r_curve
i = int(np.argmin(np.abs(px - CONF)))
rows = [dict(cls='ALL', precision_at_conf=float(P[:, i].mean()), recall_at_conf=float(R[:, i].mean()),
             map50=float(t.box.map50), map50_95=float(t.box.map), conf=CONF)]
for k, c in enumerate(t.box.ap_class_index):
    _, _, a50, a5095 = t.box.class_result(k)
    rows.append(dict(cls=best.names[int(c)], precision_at_conf=float(P[k, i]), recall_at_conf=float(R[k, i]),
                     map50=float(a50), map50_95=float(a5095), conf=CONF))
json.dump(rows, open(f'{RES}/test_metrics_v2.json', 'w'), indent=1)
with open(f'{RES}/test_metrics_v2.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
print(open(f'{RES}/test_metrics_v2.csv').read())

# Confusion matrix at the chosen threshold (same model, threshold chosen on validation)
best.val(data=DATA, split='test', imgsz=IMGSZ, conf=CONF, plots=True, project=RES, name='test_confusion_at_chosen_conf', exist_ok=True)

DS = os.path.dirname(open(DATA).read().split('path: ')[1].split('\n')[0] + '/x')
best.predict(f'{DS}/test/images', imgsz=IMGSZ, conf=CONF, save=True, project=RES, name='test_predictions_v2', exist_ok=True)

print('\nCompare with v1 (real test values from the first run): mAP50=0.602 mAP50-95=0.360 P=0.792 R=0.529 (max-F1 point)')

## Step 8 – Download the results
A ZIP downloads automatically. Keep it – it is lost when the session closes.

In [ ]:
import shutil, os, glob
from google.colab import files
out = '/content/PPE_v2_outputs'
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
wdir = os.path.dirname(os.path.dirname(WBEST))
shutil.copytree(wdir, f'{out}/winner_training_run', ignore=shutil.ignore_patterns('epoch*.pt'))
shutil.copytree('/content/results_v2', f'{out}/results_v2')
shutil.copy('/content/data.yaml', out)
for r in summary:   # keep logs of every run, not the heavy weights
    d = f"{out}/all_runs/{r['name']}"; os.makedirs(d, exist_ok=True)
    for f in ('results.csv', 'args.yaml', 'results.png'):
        p = f"{RUNS}/{r['name']}/{f}"
        if os.path.exists(p): shutil.copy(p, d)
open(f'{out}/chosen_confidence.txt', 'w').write(str(CONF))
shutil.make_archive('/content/PPE_v2_outputs', 'zip', out)
print('ZIP size (MB):', round(os.path.getsize('/content/PPE_v2_outputs.zip') / 1e6, 1))
files.download('/content/PPE_v2_outputs.zip')